# 📚 Assistant Bibliothèque Intelligent
========================================================================================
## Stack :
- LLM        : Gemini 2.0 Flash (cloud) OU Ollama llama3.2:3b (local) — switch dans l'UI
- Streaming  : générateur Python → Gradio yield
- Tool       : search_book() → Open Library API (gratuit, sans clé)
- Couvertures: Open Library Covers API
- Audio IN   : Whisper via Groq API (gratuit) OU google.genai audio
- Audio OUT  : gTTS → gr.Audio autoplay

In [14]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import json
import tempfile
import requests
from io import BytesIO
from dotenv import load_dotenv
import gradio as gr
from PIL import Image
from gtts import gTTS
from google import genai
from google.genai import types
from ollama import Client as OllamaClient

In [15]:
# ── Initialisation ────────────────────────────────────────────────────────────
load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
gemini_client  = genai.Client(api_key=GEMINI_API_KEY)
ollama_client  = OllamaClient(host="http://localhost:11434")

GEMINI_MODEL = "gemini-2.5-flash"
OLLAMA_MODEL = "llama3.2:3b"

# ── System prompt bibliothèque ────────────────────────────────────────────────
SYSTEM_PROMPT = """Tu es Léo, un bibliothécaire passionné et érudit avec 20 ans d'expérience.
Tu aides les utilisateurs à :
- Trouver des livres selon leurs goûts, humeurs ou besoins
- Découvrir des auteurs et des genres littéraires
- Obtenir des résumés, analyses et recommandations de livres
- Chercher des informations précises sur un titre ou un auteur

Ton style : chaleureux, enthousiaste pour la littérature, pédagogue.
Tu réponds toujours en français sauf si l'utilisateur parle une autre langue.
Quand un utilisateur mentionne un titre de livre spécifique ou demande une recommandation concrète,
utilise TOUJOURS l'outil search_book pour trouver les informations exactes et la couverture.
"""

In [16]:
# ── Open Library Tool ─────────────────────────────────────────────────────────
def search_book(title: str, author: str = "") -> dict:
    """
    Cherche un livre via Open Library API.
    Retourne titre, auteur, année, description et URL de couverture.
    """
    print(f"🔍 Recherche livre : '{title}' (auteur: '{author}')")
    query = f"{title} {author}".strip().replace(" ", "+")
    url   = f"https://openlibrary.org/search.json?q={query}&limit=1&fields=title,author_name,first_publish_year,cover_i,key,subject"

    try:
        resp = requests.get(url, timeout=10)
        data = resp.json()

        if not data.get("docs"):
            return {"found": False, "message": f"Aucun résultat pour '{title}'"}

        doc        = data["docs"][0]
        cover_id   = doc.get("cover_i")
        cover_url  = f"https://covers.openlibrary.org/b/id/{cover_id}-L.jpg" if cover_id else None
        subjects   = doc.get("subject", [])[:5]

        return {
            "found":        True,
            "title":        doc.get("title", title),
            "author":       ", ".join(doc.get("author_name", ["Inconnu"])),
            "year":         doc.get("first_publish_year", "?"),
            "cover_url":    cover_url,
            "cover_id":     cover_id,
            "subjects":     subjects,
            "ol_key":       doc.get("key", ""),
        }
    except Exception as e:
        return {"found": False, "message": str(e)}


def fetch_cover_image(cover_url: str) -> Image.Image | None:
    """Télécharge et retourne l'image de couverture."""
    if not cover_url:
        return None
    try:
        resp = requests.get(cover_url, timeout=10)
        if resp.status_code == 200:
            img = Image.open(BytesIO(resp.content))
            return img
    except Exception as e:
        print(f"Cover fetch error: {e}")
    return None

In [17]:
# ── Déclaration outil Gemini ──────────────────────────────────────────────────
gemini_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_book",
            description=(
                "Cherche un livre dans la base Open Library. "
                "Utilise cet outil quand l'utilisateur mentionne un titre précis, "
                "un auteur, ou demande une recommandation de livre spécifique. "
                "Retourne le titre, l'auteur, l'année et la couverture du livre."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "title": types.Schema(
                        type=types.Type.STRING,
                        description="Le titre du livre à rechercher",
                    ),
                    "author": types.Schema(
                        type=types.Type.STRING,
                        description="L'auteur du livre (optionnel)",
                    ),
                },
                required=["title"],
            ),
        )
    ]
)

In [18]:
# ── Déclaration outil Ollama ──────────────────────────────────────────────────
ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_book",
            "description": (
                "Cherche un livre dans la base Open Library. "
                "Utilise cet outil quand l'utilisateur mentionne un titre précis "
                "ou demande une recommandation concrète."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "title":  {"type": "string", "description": "Titre du livre"},
                    "author": {"type": "string", "description": "Auteur (optionnel)"},
                },
                "required": ["title"],
            },
        },
    }
]

In [19]:
AVAILABLE_TOOLS = {"search_book": search_book}

In [20]:
# ── TTS ───────────────────────────────────────────────────────────────────────
def talker(message: str) -> str:
    """Génère un MP3 via gTTS, retourne le chemin pour gr.Audio."""
    # Nettoie les emojis et markdown pour le TTS
    import re
    clean = re.sub(r"[*_`#>\[\]()]", "", message)
    clean = re.sub(r"https?://\S+", "", clean)
    clean = clean[:500]  # limite raisonnable pour le TTS

    tts = gTTS(text=clean, lang="fr")
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tts.save(tmp.name)
    return tmp.name

In [21]:
# ── Normalisation Gradio 6.x ──────────────────────────────────────────────────
def extract_text(raw) -> str:
    if isinstance(raw, list):
        return " ".join(p["text"] for p in raw if isinstance(p, dict) and "text" in p)
    return raw or ""

In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# GEMINI — streaming + tool
# ══════════════════════════════════════════════════════════════════════════════
def chat_gemini(history):
    """
    Générateur qui streame la réponse Gemini.
    Yields : (history, cover_image, audio_path)
    """
    gemini_history = []
    for msg in history[:-1]:  # tout sauf le dernier message user
        role = "model" if msg["role"] == "assistant" else "user"
        gemini_history.append(
            types.Content(role=role, parts=[types.Part.from_text(text=extract_text(msg["content"]))])
        )

    last_user_text = extract_text(history[-1]["content"])
    gemini_history.append(
        types.Content(role="user", parts=[types.Part.from_text(text=last_user_text)])
    )

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=[gemini_tool],
    )

    # Premier appel (non-streaming pour détecter tool calls)
    response   = gemini_client.models.generate_content(
        model=GEMINI_MODEL, contents=gemini_history, config=config
    )
    candidate  = response.candidates[0]
    cover_img  = None
    book_info  = None

    # ── Détection tool call ───────────────────────────────────────────────────
    fn_call = next((p.function_call for p in candidate.content.parts if p.function_call), None)

    if fn_call:
        args      = dict(fn_call.args)
        book_info = AVAILABLE_TOOLS[fn_call.name](**args)

        if book_info.get("found") and book_info.get("cover_url"):
            cover_img = fetch_cover_image(book_info["cover_url"])

        # Injecter le résultat outil
        gemini_history.append(candidate.content)
        gemini_history.append(
            types.Content(
                role="user",
                parts=[types.Part.from_function_response(
                    name=fn_call.name,
                    response=book_info,
                )],
            )
        )

        # Second appel en streaming
        stream = gemini_client.models.generate_content_stream(
            model=GEMINI_MODEL, contents=gemini_history, config=config
        )
    else:
        # Pas de tool call → on streame directement
        stream = gemini_client.models.generate_content_stream(
            model=GEMINI_MODEL, contents=gemini_history, config=config
        )

    # ── Streaming de la réponse ───────────────────────────────────────────────
    reply = ""
    for chunk in stream:
        if chunk.text:
            reply += chunk.text
            current_history = history + [{"role": "assistant", "content": reply}]
            yield current_history, cover_img, None  # audio pas encore prêt

    # Audio généré une fois la réponse complète
    audio_path = None
    try:
        audio_path = talker(reply)
    except Exception as e:
        print(f"TTS error: {e}")

    final_history = history + [{"role": "assistant", "content": reply}]
    yield final_history, cover_img, audio_path

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# OLLAMA — streaming + tool
# ══════════════════════════════════════════════════════════════════════════════
def chat_ollama(history):
    """
    Générateur qui streame la réponse Ollama.
    Yields : (history, cover_image, audio_path)
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for msg in history:
        messages.append({"role": msg["role"], "content": extract_text(msg["content"])})

    cover_img = None

    # Premier appel non-streaming pour détecter les tool calls
    response      = ollama_client.chat(model=OLLAMA_MODEL, messages=messages, tools=ollama_tools)
    assistant_msg = response.message

    if assistant_msg.tool_calls:
        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": assistant_msg.tool_calls,
        })

        for tc in assistant_msg.tool_calls:
            fn_name = tc.function.name
            fn_args = tc.function.arguments
            if fn_name in AVAILABLE_TOOLS:
                result = AVAILABLE_TOOLS[fn_name](**fn_args)
                if fn_name == "search_book" and result.get("found") and result.get("cover_url"):
                    cover_img = fetch_cover_image(result["cover_url"])
                messages.append({"role": "tool", "content": json.dumps(result)})

    # Streaming de la réponse finale
    reply  = ""
    stream = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=messages,
        tools=ollama_tools,
        stream=True,
    )

    for chunk in stream:
        token = chunk.message.content or ""
        reply += token
        current_history = history + [{"role": "assistant", "content": reply}]
        yield current_history, cover_img, None

    audio_path = None
    try:
        audio_path = talker(reply)
    except Exception as e:
        print(f"TTS error: {e}")

    final_history = history + [{"role": "assistant", "content": reply}]
    yield final_history, cover_img, audio_path

In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# Dispatcher principal
# ══════════════════════════════════════════════════════════════════════════════
def chat(history, model_choice):
    if model_choice == "🌐 Gemini 2.0 Flash":
        yield from chat_gemini(history)
    else:
        yield from chat_ollama(history)

In [25]:
# ── Transcription audio (micro) ───────────────────────────────────────────────
def transcribe_audio(audio_path):
    """
    Transcrit l'audio via Gemini (pas besoin de clé supplémentaire).
    Retourne le texte transcrit.
    """
    if audio_path is None:
        return ""
    try:
        with open(audio_path, "rb") as f:
            audio_bytes = f.read()

        import base64
        audio_b64 = base64.b64encode(audio_bytes).decode()

        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                types.Content(parts=[
                    types.Part.from_bytes(data=audio_bytes, mime_type="audio/wav"),
                    types.Part.from_text("Transcris exactement ce qui est dit dans cet audio. Réponds uniquement avec la transcription, sans commentaire."),
                ])
            ],
        )
        return response.text.strip()
    except Exception as e:
        print(f"Transcription error: {e}")
        return ""

In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# Interface Gradio — thème bibliothèque vintage
# ══════════════════════════════════════════════════════════════════════════════
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,400;0,700;1,400&family=Crimson+Text:ital@0;1&display=swap');

:root {
    --parchment:  #f5efe0;
    --leather:    #8b4513;
    --gold:       #c9a84c;
    --ink:        #2c1810;
    --sage:       #6b7c5e;
    --cream:      #fdf6e3;
}

body, .gradio-container {
    background-color: var(--parchment) !important;
    font-family: 'Crimson Text', Georgia, serif !important;
    color: var(--ink) !important;
}

#title-block {
    text-align: center;
    padding: 2rem 1rem 1rem;
    border-bottom: 3px double var(--gold);
    margin-bottom: 1.5rem;
}

#title-block h1 {
    font-family: 'Playfair Display', serif !important;
    font-size: 2.4rem !important;
    color: var(--leather) !important;
    letter-spacing: 0.05em;
    margin: 0;
}

#title-block p {
    font-family: 'Crimson Text', serif !important;
    font-style: italic;
    color: var(--sage) !important;
    font-size: 1.1rem;
    margin-top: 0.3rem;
}

.gr-button-primary {
    background: var(--leather) !important;
    border: none !important;
    font-family: 'Playfair Display', serif !important;
    letter-spacing: 0.05em;
}

.gr-button-secondary {
    background: var(--sage) !important;
    border: none !important;
}

#chatbot {
    background: var(--cream) !important;
    border: 1px solid var(--gold) !important;
    border-radius: 4px !important;
}

#cover-display {
    border: 3px solid var(--gold) !important;
    border-radius: 4px !important;
    background: var(--cream) !important;
}

footer { display: none !important; }
"""

with gr.Blocks(css=CSS, title="📚 Léo — Bibliothécaire IA") as ui:

    # ── En-tête ───────────────────────────────────────────────────────────────
    gr.HTML("""
    <div id="title-block">
        <h1>📚 Léo — Bibliothécaire IA</h1>
        <p>Votre guide littéraire personnel · Powered by Gemini & Ollama</p>
    </div>
    """)

    with gr.Row():
        # ── Colonne gauche : chat ─────────────────────────────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=480,
                label="Conversation avec Léo",
                elem_id="chatbot",
                avatar_images=(None, "https://api.dicebear.com/7.x/bottts/svg?seed=leo"),
            )

            with gr.Row():
                entry = gr.Textbox(
                    placeholder="Demandez un livre, un auteur, une recommandation...",
                    label="",
                    scale=5,
                    container=False,
                )
                send_btn = gr.Button("Envoyer →", scale=1, variant="primary")

            with gr.Row():
                audio_input = gr.Audio(
                    sources=["microphone"],
                    type="filepath",
                    label="🎙️ Parler à Léo",
                    scale=3,
                )
                transcribe_btn = gr.Button("📝 Transcrire", scale=1, variant="secondary")

            with gr.Row():
                model_selector = gr.Radio(
                    choices=["🌐 Gemini 2.5 Flash", "🦙 Ollama llama3.2:3b"],
                    value="🌐 Gemini 2.5 Flash",
                    label="Modèle LLM",
                )
                clear_btn = gr.Button("🗑️ Effacer", variant="secondary")

        # ── Colonne droite : couverture + audio ───────────────────────────────
        with gr.Column(scale=2):
            cover_output = gr.Image(
                height=400,
                label="📖 Couverture",
                elem_id="cover-display",
            )
            audio_output = gr.Audio(
                label="🔊 Réponse vocale",
                autoplay=True,
                visible=True,
            )
            gr.HTML("""
            <div style="text-align:center; margin-top:1rem; font-family:'Crimson Text',serif;
                        font-style:italic; color:#6b7c5e; font-size:0.95rem;">
                « Un livre est un jardin qu'on porte dans sa poche. »<br>
                <small>— Arab proverb</small>
            </div>
            """)

    # ── Message de bienvenue ──────────────────────────────────────────────────
    WELCOME = [{"role": "assistant", "content": (
        "Bonjour et bienvenue ! 📚 Je suis Léo, votre bibliothécaire IA.\n\n"
        "Je peux vous aider à :\n"
        "- **Trouver un livre** selon vos goûts ou votre humeur\n"
        "- **Découvrir des auteurs** et des genres littéraires\n"
        "- **Obtenir un résumé** ou une analyse d'un ouvrage\n"
        "- **Recommander des lectures** personnalisées\n\n"
        "Que puis-je faire pour vous aujourd'hui ? 🌟"
    )}]

    # ── Logique des interactions ──────────────────────────────────────────────
    def do_entry(message, history):
        if not message.strip():
            return "", history
        history = history + [{"role": "user", "content": message}]
        return "", history

    def do_transcribe(audio_path, current_text):
        if not audio_path:
            return current_text
        transcribed = transcribe_audio(audio_path)
        return transcribed if transcribed else current_text

    # Envoi via bouton ou Enter
    send_event = (
        entry.submit(do_entry, [entry, chatbot], [entry, chatbot])
             .then(chat, [chatbot, model_selector], [chatbot, cover_output, audio_output])
    )
    send_btn.click(do_entry, [entry, chatbot], [entry, chatbot]).then(
        chat, [chatbot, model_selector], [chatbot, cover_output, audio_output]
    )

    # Transcription micro → textbox
    transcribe_btn.click(do_transcribe, [audio_input, entry], [entry])

    # Clear
    clear_btn.click(
        lambda: (WELCOME, None, None),
        outputs=[chatbot, cover_output, audio_output],
        queue=False,
    )

    # Initialisation avec message de bienvenue
    ui.load(lambda: WELCOME, outputs=[chatbot])

ui.launch(inbrowser=True)

/tmp/ipykernel_47547/1888098300.py:72: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="📚 Léo — Bibliothécaire IA") as ui:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


🔍 Recherche livre : '{'author': null, 'title': null}' (auteur: '')
